## GPT2 implementation


KV cache


In [ ]:
class Cache:

	def __init__(self, n, max_seq_len, h):
		pass

Normalization techniques

- batchnorm
- layernorm
- RMSnorm
- pre- or post-LayerNorm


In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module): 

	def __init__(self, d_embed:int, eps:float=1e-4): 
		
		super().__init__()
		self.eps = eps
		
		self.weight = nn.Parameter(d_embed)

	def forward(self, x_in: torch.Tensor): 

		variance = x_in.pow(2).mean(dim=-1, keepdim=True) # mean sum of squares along feature dimension, (B, T, D)

		x_rms = torch.rsqrt(variance + self.eps) # (B, T, D)

		return x_in * x_rms * self.weight

### NOTE: pre- and post-LN are just whether you place it before the layer, or after the layer + residual stream
class LayerNorm(nn.Module):

	def __init__(self, d_embed: int, eps:float=1e-5):

		super().__init__()

		self.eps = eps
		self.weight = nn.Parameter(torch.ones(d_embed))
		self.bias = nn.Parameter(torch.zeros(d_embed))

	def forward(self, x_in):

		b, t, d = x_in.shape
		
		variance = x_in.var(dim=-1, keepdim=True) # not using Bessel's correction
		mean = x_in.mean(dim=-1, keepdim=True)

		x_out = (x_in - mean) * torch.rsqrt(variance + self.eps) # reciprocal square root

		return x_out * self.weight + self.bias

class BatchNorm(nn.Module):

	def __init__(self, d_embed:int, eps:float=1e-5, momentum:float=0.1): 
		
		super().__init__()

		self.eps = eps
		self.d_embed = d_embed
		self.momentum = momentum

		self.weight = nn.Parameter(torch.ones(d_embed))
		self.bias = nn.Parameter(torch.zeros(d_embed))

		self.register_buffer('running_mean', torch.zeros(d_embed))
		self.register_buffer('running_var', torch.zeros(d_embed))

	def forward(self, x_in:torch.Tensor, training:bool=False): 

		b, t, d = x_in.shape
		x_in = x_in.flatten(0,1)
		
		if training: 
			
			variance = x_in.var(dim=0, keepdim=True, unbiased=True) # (b*t, d)
			mean = x_in.mean(dim=0, keepdim=True)

			with torch.no_grad():
				self.running_mean.mul_(1-self.momentum).add_(self.momentum*mean)
				self.running_var.mul_(1-self.momentum).add_(self.momentum*x_in.var(dim=0, keepdim=True, unbiased=False)) # since here we are actually estimating, we use Bessel's

		else: 

			mean, variance = self.running_mean, self.running_var
		
		x_out = (x_in - mean) * torch.rsqrt(variance + self.eps)
		x_out = x_out.view(b, t, -1)

		return x_out * self.weight + self.bias

Residual connections

- implementing a residual stream
- implementing ByteDance HC
- implementing Deepseek mHC


Learning rate scheduler

- cosine decay


Optimizer

- AdamW
- MUON
- SHAMPOO


Dropout

_dropout as ensemble learning_

- dropout acts as ensemble learning during inference. During training time, say we set dropout = 0.99, thus we train lots of different sub-networks, each lighting up a different set of neurons. Each of these neurons learns to produce the output, independent of each other, or if there's neurons shared between subnetworks, the neuron becomes a shared parameter, but this weakens the ensemble learning since if there's a lot of shared neurons, your subnetworks are not producing meaningfully different outputs.
- at inference time, we use .eval() to turn off dropout. Now each subnetwork is activated in the forward pass, and the activations are a combination of all the subnetworks votes on what the right activation is, since each has learned a different 'correct answer' for the output vector.

_dropout during self-attention_

- we place dropout in 2 different places
  - after softmax produces the scores
  - right before we merge with the layer output with the residual stream
- why there?
  - we don't want to dropout the residual stream, rather we care about the weights in the layer, learning to develop subnetworks. The residual stream changes on every token - this is not something the model needs to learn

  - models can become overly reliant and attend only to the tokens immediately adjacent to the next token (e.g. the cat chases the dog. It was very fast. It refers to the \_**\_ -> the model may learn, when predicting the \_\_** to attend only to the closest tokens, so 'dog' when 'it' may refer to the cat)


Implement GPT2 from scratch


In [ ]:
import re
import torch
from collections import defaultdict
from torch.utils.data.dataloader import DataLoader
from torch.utils.data.dataset import Dataset
from torch.utils.data import TensorDataset

# tokenize the data
class Tokenizer:

	def __init__(self): 
		
		# initialize a dictionary that is the 256 unicode characters
		self.vocab = {idx: bytes([idx]) for idx in range(256)} # bytes([4]) creates a bytes object for the value 4, while bytes(4) creates a bytes object of length 4 - initialized with zeros
		self.merges = defaultdict(int)

		self.special_tokens = {} # str -> int
		self.inverse_special = {} # int -> str

	@property
	def vocab_size(self):
		return len(self.vocab) # 256 bytes + merges + specials

	def get_stats(self, ids):

		# given a sequence find the most common occuring pairs
		counts = defaultdict(int)
		for pair in zip(ids, ids[1:]): 
			counts[pair] += 1 # initializes a key of [id1, id2] and then adds to the count

		return counts

	def merge_seq(self, ids, pair, idx):

		new_ids = []
		i = 0

		while i < len(ids):
			if i < len(ids)-1 and ids[i]==pair[0] and ids[i+1]==pair[1]:
				i += 2
				new_ids.append(idx)
			else: 
				new_ids.append(ids[i])
				i += 1

		return new_ids

	def merge(self, ids, num_merges): 

		for i in range(num_merges): 

			# find the most common occuring pair
			stats = self.get_stats(ids)
			if not stats: # sequence collapsed to <2 tokens, nothing left to merge
				break
			freq_pair = max(stats, key=stats.get) 

			# with the most frequent pair, merge the sequence and add it to the merges list
			token_id = 256 + i
			ids = self.merge_seq(ids, freq_pair, token_id) # newly merged sequence
			self.merges[freq_pair] = token_id # add this to the merges dictionary 

		for (p0, p1), idx in self.merges.items(): 

			self.vocab[idx] = self.vocab[p0] + self.vocab[p1] # p0,p1 = 145, 165, the new vocab idx = 257, so now vocab[257] = bytes(145) + bytes(165) (concat - since we indexed into vocab,s grabbed the bytes and concatenated)

		return ids

	def register_special_tokens(self, tokens):

		# call AFTER merge(), so these ids sit above every merged token
		for tok in tokens:
			idx = len(self.vocab)
			self.special_tokens[tok] = idx
			self.inverse_special[idx] = tok
			self.vocab[idx] = tok.encode('utf-8') # mirror into vocab, else decode() KeyErrors on it

	def _encode(self, text):

		ids = text.encode('utf-8') # returns array of bytes

		for pair, idx in self.merges.items():
			ids = self.merge_seq(ids, pair, idx)

		return ids
	
	def encode(self, text, allowed_special=True):

		# allowed_special=False for untrusted text - then a literal "<|endoftext|>" tokenizes as
		# ordinary bytes and cannot forge a document boundary
		if not allowed_special or not self.special_tokens:
			return self._encode(text)

		# capturing group keeps the delimiters in the split output; re.escape because
		# "<|endoftext|>" contains |, which is regex alternation
		pattern = "(" + "|".join(re.escape(t) for t in self.special_tokens) + ")"

		ids = []
		for chunk in re.split(pattern, text):
			if chunk in self.special_tokens:
				ids.append(self.special_tokens[chunk])
			elif chunk:
				ids.extend(self._encode(chunk))

		return ids

	def decode(self, ids):

		new_ids = [self.vocab[int(idx)] for idx in ids] # int() so numpy/tensor scalars work too
		tokens = b"".join(new_ids)
		text = tokens.decode('utf-8', errors='replace') # replace: a batch slice can cut a multi-byte char in half
		
		return text

tk = Tokenizer()

**Dataloader**

- dataloader that loads from external url (online dataset / hf)
- dataloader that takes in ids and converts to (B, T, D)


In [ ]:
import requests
import os

cdir = os.path.abspath('')
data_path = os.path.join(cdir, "data")

os.makedirs(data_path, exist_ok=True) # make directory if doesn't exist
input_file_path = os.path.join(data_path, "input.txt")

# input_file_path = os.path.join(os.path.dirname(__file__), 'input.txt') # only works in .py files

if not os.path.exists(input_file_path): # if nothing exists at the file path right now
	data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
	with open(input_file_path, 'w') as f: # open at the input file path, with write permissions - creating a file since it doesn't exist, naming it as f
		f.write(requests.get(data_url).text)

with open(input_file_path, 'r') as f:
	text = f.read()

# TRAIN the tokenizer once, on the raw bytes. num_merges is small here so this runs in seconds -
# the naive get_stats-over-the-whole-sequence loop is O(n) per merge, so 50k merges on 1MB of
# text would take hours. real BPE trainers keep an incremental pair-count index instead.
tk.merge(list(text.encode('utf-8')), num_merges=500)

# <|endoftext|> does three jobs at once: document separator, BOS, and EOS. they're all the same
# signal - "a document boundary is here" - which is why GPT-2 needs no separate BOS or PAD token
tk.register_special_tokens(["<|endoftext|>"])
eot = tk.special_tokens["<|endoftext|>"]

# tinyshakespeare is one continuous file, so there's a single document here. splitting on blank
# lines would treat each scene as its own doc - shown as the pattern you'd use on a real corpus
documents = [text]

ids = []
for doc in documents:
	ids.append(eot) # LEADING eot, so the very first window opens on a boundary token (the BOS role)
	ids.extend(tk.encode(doc, allowed_special=False)) # False: the separator is ours to insert, not the document's

print(f"vocab_size={tk.vocab_size}, {len(text)} chars -> {len(ids)} tokens ({len(text)/len(ids):.2f}x compression)")

In [ ]:
import torch
import numpy as np

# one flat 1-D stream of token ids. this is WHY pretraining needs no PAD token: we slice fixed
# windows out of a continuous stream, so every window is exactly max_seq_len by construction and
# nothing is ever ragged. padding only matters for variable-length data (fine-tuning, batched
# inference on different-length prompts)
data_ids = torch.tensor(ids, dtype=torch.long)

training_split = int(len(data_ids) * 0.9)

train_data, test_data = data_ids[:training_split], data_ids[training_split:] # training_split = 0.9

# uint16 holds 0..65535. fine for GPT-2's 50257, but it wraps SILENTLY if the vocab ever exceeds
# it - assert so it fails loudly instead of corrupting the .bin files
assert tk.vocab_size <= 65535, f"vocab_size {tk.vocab_size} overflows uint16 - switch to uint32"

train_ids = np.asarray(train_data, dtype=np.uint16) # does not copy in memory
test_ids = np.asarray(test_data, dtype=np.uint16)

train_ids.tofile(os.path.join(data_path, 'train.bin'))
test_ids.tofile(os.path.join(data_path, 'test.bin'))

def get_batch(split, batch_size=64, max_seq_len=1024):
	d = train_data if split == 'train' else test_data

	# index into random integers into the train_data, do this along the batch_size dimension
	# we want any integers before (length of the data - longest sequence)
	# the -1 matters: y reads one position PAST x's end, so without it the last valid index overruns
	ix = torch.randint(len(d)-max_seq_len-1, (batch_size,)) # (batch_size,)
	x = torch.stack([d[i: i+max_seq_len] for i in ix])
	y = torch.stack([d[i+1: i+max_seq_len+1] for i in ix])
	return x,y # train, targets

# NOTE: don't assign this back to train_data - that would overwrite the full corpus with a single
# batch, and every later get_batch would slice from the batch instead of the data
batch, targets = get_batch('train')
print(batch.shape, targets.shape)
print(tk.decode(batch[0, :100].tolist()))

**Swiglu activation function**
$$SwiGLU(x)=W_{down}[SiLU(W_{gate}x)⊙(W_{up}x)]$$

$$ SiLU(x) = \dfrac{x}{1+e^{-x}}$$

To ensure the amount of parameters (memory) for a SWIGLU = normal FFN, we compress the hidden dim

- normal FFN: fan*in * hidden*dim + fan_out * hidden_dim = 2dm (d = fan_in ~ fan_out, m = hidden_dim)
- SWIGLU: 3dm, so we make m_SWIGLU = 2/3 m_FFN
  - m_FFN = 4\*d typically
  - m_SWIGLU = 8/3 \* d (smaller hidden_dim)

Experts are typically themselves just one SWIGLU layer (Deepseek, Mistral, Qwen)


**Simplified progression of activation functions**

Sigmoid / tanh -> RELU -> ELU -> GELU + Layer norm -> GLU -> SwiGLU

- RELU - sparsity + no more vanishing gradients
- ELU - no more dead neurons, zero mean-centering for activations to prevent magnitude explosion from RELU (assuming activations 0 ≤ x ≤ 1 or 2)
- GELU + Layernorm - Layer normalization fixed zero-mean centering and better, GELU brings back RELU with smoothness vs. kinked to prevent dead neurons
- GLU - gating, meaning one determines how much of this token's info to pass on, while the other determines what to pass on (similar to attention!)
- SwiGLU - better gating function

<img src="../assets/activ-fx.png">
<img src="https://sebastianraschka.com/images/blog/2025/from-gpt-2-to-gpt-oss/8.png" width="400">


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

def silu(x_in):

	return x_in / (1+torch.exp(x_in))

class SwiGLU(nn.Module):

	def __init__(self, d_embed:int):

		super().__init__()

		self.d_embed = d_embed
		self.d_hidden = int(8/3 * d_embed) # GPT-OSS uses 2x instead of 8/3
		self.fc1 = nn.Linear(d_embed, self.d_hidden, bias=False)
		self.fc2 = nn.Linear(d_embed, self.d_hidden, bias=False)
		self.fc3 = nn.Linear(self.d_hidden, d_embed, bias=False)

	def forward(self, x_in):

		# (SILU(W_g @ x) * W_down @ x) @ W_up 
		
		return self.fc3(silu(self.fc1(x_in)) * self.fc2(x_in))

x_in = torch.randn(32,1024,256)

swig = SwiGLU(d_embed=256)

out = swig(x_in)
out.shape

torch.Size([32, 1024, 256])

**Transformer architecture (Deepseek-v3)**

_Simple implementation_

- Embedding layer
  - 7,168 d_embed
  - vocab_size = 129k
- Transformer block x61
  - RMSNorm 1 (pre-norm)
  - MLA + RoPE embeddings
  - 128 heads
  - max_seq_len 128k tokens
  - Residual connection 1
  - RMSNorm 2 (pre-norm)
  - MoE
  - First 3 use dense FFN with hidden of 18,432 instead of MoE
    - 256x SwiGLU experts
  - 8x active, 1x shared
  - Residual connection 2
- Final RMSNorm
- LM head

<img src="https://substackcdn.com/image/fetch/$s_!W4Qo!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Ff4b97110-d705-4531-b4af-4f87187a8dea_1393x1394.png" width="400">


In [ ]:
class Transformer(nn.Module):

	def __init__(self, d_embed):
		
		self.d_embed = d_embed